# Klar v2.6 — auditable Qwen3.5-9B adapter feasibility run

This Kaggle notebook runs one tiny, synthetic, text-only LoRA experiment for
either the **Precision** or **Writer** slot. It is an engineering proof, not a
quality claim and not a production training recipe.

The built-in records are authored synthetic fixtures. They contain no
production résumé, application, recruiter-message, user-note, diagnostic or
cloud-provider content. The held-out fixtures are never passed to training.

Before running: enable a Kaggle **T4** GPU and internet access. This copy is set to
**ADAPTER_SLOT = "writer"**. Run it in a freshly restarted session — switching
slots without a restart trains one slot's fixtures on top of the other slot's
adapter. Archives are now named per slot, restored step and weight hash, so the
two slots cannot be confused in a downloads folder. The model, dataset and Python dependencies are pinned and
the run is seeded, but Kaggle's base image and GPU can change.
The provenance file records the observed environment; do not claim bit-for-bit
reproducibility across different Kaggle sessions.

**Fix log — 2026-08-09.** Transformers v5 deprecated `warmup_ratio` in favour of
`warmup_steps` and removed it in v5.2, so the pinned Transformers commit no
longer accepts it. `warmup_steps` now takes a float below 1 and means exactly
what `warmup_ratio` used to mean: a fraction of the total training steps. The
setup cell now selects whichever keyword the installed Transformers actually
exposes, and checks every `SFTConfig` keyword the training cell will use before
the 9B download instead of after it.

**Review follow-ups — 2026-08-09.** Four changes after reviewing the first
Precision and Writer artifacts. The held-out cell now also generates with the
adapter switched off, so every output has a base-model control beside it and the
run can show what the training actually changed. The training metrics and the
per-step log history are written into `provenance.json` instead of only being
printed. The compute dtype is chosen from the GPU's compute capability rather
than from `torch.cuda.is_bf16_supported()`, which defaults to
`including_emulation=True` and reports True on a T4 that has no bf16 tensor
cores. And `adapter_config.json` is rewritten with `target_modules` sorted,
because PEFT serialises that field from a set and two identical configurations
would otherwise hash differently.

**Probe v5, Precision — 2026-08-09.** Switching back to the Precision slot
exposed a gap. Language detection abstains on compact JSON — correctly, since
there are too few function words to classify — which left this slot with only
the degeneracy score, and degeneracy is a blunt instrument: on the measured
60-step curve it first fired at step 50, by which point the German output had
been malformed for twenty steps. The stutter `"employment":"employment":` begins
at step 30 and scores only 0.0333. A structural check now runs alongside: if a
fixture's target parses as JSON, its generation must parse as JSON too. It is
driven by the fixture data rather than by ADAPTER_SLOT, so writer prose is
exempt automatically and a mixed slot would need no new flag. Markdown fences
are unwrapped before parsing and recorded as `usedCodeFence` rather than failed,
since fenced JSON is untidy rather than broken. Validated against every
generation in both artifacts: it flags the German row from step 30 onward,
passes all Precision outputs that are genuinely well-formed including the fenced
base-model controls, and never engages on writer prose.

**Probe v4 — 2026-08-09.** The v3 run shipped a broken adapter stamped
`promotable: true`. The lowest held-out loss fell on step 16, which was not on
the every-five generation grid, so no guard ever ran there, and `verdict()` read
the absent result as a pass — the artifact's own post-training generations came
back German. The guard was right; the selection logic around it was not. Four
changes. Generation now also runs on every step that sets a new loss minimum, so
any step that could be kept is checked. Two bests are tracked: `best_any` drives
patience, `best_clean` is the lowest loss among steps whose guards ran and
passed, and only `best_clean` is ever snapshotted or restored. `guardsPassed` is
explicitly `None` when the guards did not run, so an absent result can no longer
be mistaken for a pass. And after restoring, both guards run again on the exact
weights about to be hashed — `verdict()` now reports that check rather than a
step from the training history, `promotable` is lifted to the top level of
provenance.json, and a failing artifact gets `-NOT-PROMOTABLE` in its filename.

**Probe v3 — 2026-08-09.** The writer run improved held-out loss by 56% and
still answered an English prompt in German, reproducing 23 characters of
writer-de-01's target verbatim. Teacher-forced loss cannot see that, because
forcing the correct prefix hides a first-token divergence; the degeneracy score
cannot either, because the output is fluent German scoring 0.0. Four changes.
Per-row held-out loss, since the aggregate hid that one row began at 1.3980 and
the other at 0.5752 and that the gain was concentrated in the broken one. A
function-word language guard that abstains on JSON — validated against every
generation in both artifacts: it flags the German answer, agrees with the
fixture on all fifteen correct outputs, and returns None on all twelve Precision
JSON outputs. Completion masking now built from the chat template with the
prefix relationship asserted, replacing string concatenation plus a guessed EOS
and an assumption that re-tokenising a join preserves the prompt's own token
boundary, which BPE does not guarantee. And `EARLY_STOP_PATIENCE` 5 to 8 with
`EARLY_STOP_MIN_DELTA` 1e-3 to 1e-4, after the writer run stopped at step 15 on
a curve that was falling again and had missed the improvement threshold by a
thousandth. Also fixed: stopping on a step off the generation grid used to
append a duplicate history entry and count the patience twice.

**Writer slot — 2026-08-09.** Switched to the writer slot after the Precision
run early-stopped at step 15, kept step 10, and wrote a step-10 adapter whose
hash differs from the step-60 one. Three things were checked against the writer
fixtures before switching rather than assumed: the slot has 6 train and 2 eval
records so `gradient_accumulation_steps = 6` still divides exactly; all eight
writer targets score 0.0 on the degeneracy metric, so its threshold carries over
untouched; and writer targets reach 332 characters against 188 for precision, so
`PROBE_MAX_NEW_TOKENS` is now slot-aware and gives prose 192 tokens. Read
`matchesExpected` as noise on this slot — exact string match on prose is not a
quality signal.

**Early stop and best checkpoint — 2026-08-09.** The probe's first run measured
held-out loss bottoming at 0.5372 on step 10 and climbing to 1.1065 by step 60,
while training loss fell to 8.2e-05 — so fifty of the sixty steps made the
adapter worse on data it had not seen. It also showed the German repetition loop
arriving progressively (absent at step 10, key leaked at 15, doubled at 45,
looping at 50 and 60), which rules out emulated bf16 and leaves memorisation as
the cause. The probe now snapshots the trainable tensors at every new held-out
minimum, stops training after `EARLY_STOP_PATIENCE` probes without improvement,
and `restore_best()` writes that snapshot back before the held-out cell or the
artifact sees the model. `MAX_STEPS` stays at 60 on purpose: it sets the cosine
schedule, so lowering it changes the learning-rate path and would invalidate the
measured step-10 optimum. Early stopping, not a smaller ceiling, is what makes
the run short.

**Held-out probe — 2026-08-09.** The first Precision artifact recorded exactly
one held-out measurement, taken after the last step, and it showed a German
generation collapsed into a repetition loop. A single end-of-run sample cannot
separate an adapter that memorised six fixtures from a numerical problem in the
emulated bf16 path, and the two have opposite fixes. A new cell before the
training cell adds a `HeldOutProbe` callback: teacher-forced held-out loss every
step, held-out generation every fifth step scored for degeneracy, and a
base-model control captured once at step 0. Both series are written into
`provenance.json`. A rising held-out loss against a falling training loss is
overfitting and names the step the run should have stopped at.

**Precision follow-up — 2026-08-09.** Choosing fp16 on a T4 turns on
`torch.amp.GradScaler`, and the scaler's unscale kernel
(`_amp_foreach_non_finite_check_and_unscale_cuda`) is implemented for float32
and float16 gradients only — a bfloat16 trainable tensor makes the first
optimizer step raise `NotImplementedError`. Something between `get_peft_model`
and the training loop puts the adapters back into the base model's dtype, so
the float32 upcast in the model cell is no longer trusted on its own: the
training cell re-asserts it once at construction time and again from a
`TrainerCallback` at `on_train_begin`, which runs after accelerate has prepared
the model. `PRECISION_MODE` in the model cell is the escape hatch — set it to
`"bf16"` to force the emulated bfloat16 path, which needs no GradScaler at all
and therefore cannot hit this limitation.

**Schedule follow-up — 2026-08-09.** The first fp16 run learned nothing for ten
of its twelve steps. GradScaler starts at `init_scale=65536` and halves it on
every overflow while skipping the optimizer step, and Transformers only advances
the learning-rate schedule when the optimizer actually ran — so the scaler spent
nine steps calibrating with the schedule frozen, and only the last two steps
trained. That cost is invisible in a thousand-step run and fatal in a twelve-step
one, so `PRECISION_MODE` now defaults to `"bf16"`, which has no scaler and
therefore no skipped steps.

Two shape fixes went with it. `gradient_accumulation_steps` is now 6 rather than
4: Transformers derives updates per epoch as
`len(dataloader) // gradient_accumulation_steps`, and 6 // 4 = 1 left two of the
six fixtures in a ragged tail. Six fixtures with accumulation 6 makes every
optimizer step one clean full-batch pass over the whole slot, which also makes
dataset order irrelevant. And `MAX_STEPS` rises from 12 to 60, because twelve
steps only ever produced two real updates. The extra steps buy a readable
training curve, not quality: on six fixtures this is memorisation by design and
must not be read as evidence of generalisation.


In [ ]:
# Fail before downloading the model if Kaggle's managed image is too old for
# the pinned Qwen3.5/Transformers stack. This was checked against Kaggle GPU
# image v170 (PyTorch 2.10.0 + torchvision 0.25.0) on 2026-07-31.
import os

# The CUDA caching allocator reads this when it makes its first allocation, so
# it has to be set before anything touches the GPU. Without it a 16 GB card can
# hold around 2 GiB reserved but unallocated, and one large contiguous request
# then fails even though the free total looks sufficient.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

from packaging.version import Version
import torch
import torchvision

torch_version = Version(torch.__version__.split("+")[0])
if torch_version < Version("2.10.0"):
    raise RuntimeError(
        f"PyTorch {torch.__version__} is too old. In Kaggle, choose the current "
        "notebook image, restart the session, and run again. Do not replace "
        "Kaggle's CUDA-enabled PyTorch with an arbitrary CPU wheel."
    )

if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU in Kaggle notebook settings, then restart the session.")

compute_capability = torch.cuda.get_device_capability(0)
if compute_capability < (7, 0):
    raise RuntimeError(
        f"GPU compute capability {compute_capability} is too old for the pinned "
        "CUDA 12.8/bitsandbytes stack. Choose an Nvidia T4 (not P100) in "
        "Kaggle notebook settings, restart the session, and run again."
    )

print({
    "kaggle_torch": torch.__version__,
    "kaggle_torchvision": torchvision.__version__,
    "gpu": torch.cuda.get_device_name(0),
    "compute_capability": compute_capability,
})


In [ ]:
# Reproducible dependency snapshot captured 2026-07-31.
# Qwen3.5 requires a current Transformers build, so Git dependencies are pinned
# to immutable commits instead of a moving main branch.
%pip install -q --disable-pip-version-check \
  "transformers @ git+https://github.com/huggingface/transformers.git@71c6f699ac9b3f8fc42a6a3e9dc59034c349a678" \
  "peft @ git+https://github.com/huggingface/peft.git@9f1fe21d8131a24634d6d23c13efa2aae72b6cca" \
  "trl @ git+https://github.com/huggingface/trl.git@922dc584664d87482935e1fa7d958930fc5223cf" \
  bitsandbytes==0.50.0 datasets==5.0.1 accelerate==1.14.0 safetensors==0.8.0


In [ ]:
import hashlib
import inspect
import json
import os
import platform
import random
import shutil
import subprocess
import zipfile
from pathlib import Path

import numpy as np
import torch
from packaging.version import Version
import accelerate
import bitsandbytes
import datasets
import peft
import safetensors
import transformers
import trl
from datasets import Dataset
from peft import LoraConfig, PeftModel, get_peft_model
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainerCallback,
    set_seed,
)
from trl import SFTConfig, SFTTrainer

DEPENDENCY_REVISIONS = {
    "transformers": "71c6f699ac9b3f8fc42a6a3e9dc59034c349a678",
    "peft": "9f1fe21d8131a24634d6d23c13efa2aae72b6cca",
    "trl": "922dc584664d87482935e1fa7d958930fc5223cf",
}

# SFTConfig subclasses transformers.TrainingArguments, so a Transformers
# release that drops a training keyword drops it from SFTConfig as well.
# Transformers v5 deprecated `warmup_ratio` in favour of `warmup_steps` and
# removed it in v5.2 (huggingface/transformers#41326). On v5, `warmup_steps`
# accepts a float below 1 and reads it as a fraction of the total training
# steps, which is what `warmup_ratio` used to mean. Select the keyword the
# installed version exposes instead of hard-coding either spelling.
TRANSFORMERS_VERSION = Version(transformers.__version__.split("+")[0])
WARMUP_FRACTION = 0.1
WARMUP_ARGUMENT = (
    "warmup_steps" if TRANSFORMERS_VERSION >= Version("5.0") else "warmup_ratio"
)

# Every keyword the training cell will hand to SFTConfig. A dataclass raises
# TypeError on one unexpected keyword at a time, so validating the whole set
# here reports all of them together, and reports them in seconds instead of
# after a multi-gigabyte model download.
required_sft_config = {
    "output_dir", "max_length", "packing", "max_steps",
    "per_device_train_batch_size", "gradient_accumulation_steps",
    "learning_rate", WARMUP_ARGUMENT, "lr_scheduler_type", "optim",
    "logging_steps", "save_strategy", "eval_strategy", "report_to",
    "gradient_checkpointing", "completion_only_loss", "full_determinism",
    "bf16", "fp16", "seed", "data_seed",
}
required_sft_trainer = {
    "model", "args", "train_dataset", "processing_class",
}

# The held-out cell needs to generate once with the adapter active and once
# with it switched off. Fail here rather than after training if this PEFT
# build cannot toggle the adapter.
if not hasattr(PeftModel, "disable_adapter"):
    raise RuntimeError(
        "This PEFT build has no PeftModel.disable_adapter, so the held-out "
        f"cell cannot produce a base-model control. peft=={peft.__version__}"
    )
missing_config = required_sft_config - set(inspect.signature(SFTConfig).parameters)
missing_trainer = required_sft_trainer - set(inspect.signature(SFTTrainer).parameters)
if missing_config or missing_trainer:
    raise RuntimeError({
        "transformers": transformers.__version__,
        "trl": trl.__version__,
        "missing_sft_config_parameters": sorted(missing_config),
        "missing_sft_trainer_parameters": sorted(missing_trainer),
    })

cuda_probe = (torch.ones(1, device="cuda") * 2).item()
torch.cuda.synchronize()
if cuda_probe != 2:
    raise RuntimeError("The Kaggle GPU failed a real CUDA tensor operation.")

SEED = 260731
BASE_MODEL = "Qwen/Qwen3.5-9B"
BASE_REVISION = "c202236235762e1c871ad0ccb60c8ee5ba337b9a"
# Beginner-visible experiment switch. Use "precision" for the first clean
# session, then "writer" in a newly restarted session. Restarting matters: the
# model cell has already put 9B of weights and a slot-specific adapter on the
# GPU, and switching this string without a restart trains the writer fixtures
# on top of the precision adapter.
ADAPTER_SLOT = "writer"
if ADAPTER_SLOT not in {"precision", "writer"}:
    raise ValueError('ADAPTER_SLOT must be "precision" or "writer"')

OUTPUT_DIR = Path("/kaggle/working") / f"klar-{ADAPTER_SLOT}-adapter-v1"
MAX_LENGTH = 1024
# A ceiling, not a target. MAX_STEPS feeds the cosine learning-rate schedule as
# well as the stopping condition, so lowering it to the measured optimum would
# change the learning-rate path the optimum was measured under and the number
# would no longer mean what it meant. Holding it at 60 reproduces the schedule
# of the run that found the step-10 minimum; EARLY_STOP_PATIENCE in the probe
# cell is what actually ends the run, near step 15 on the measured curve.
# Raise this only alongside a rerun of the probe.
MAX_STEPS = 60
# Must divide the number of training fixtures exactly; see the training cell.
GRADIENT_ACCUMULATION_STEPS = 6

os.environ["TOKENIZERS_PARALLELISM"] = "false"
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
set_seed(SEED)
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

if not torch.cuda.is_available():
    raise RuntimeError("This feasibility notebook requires a Kaggle GPU.")

print({
    "slot": ADAPTER_SLOT,
    "warmup_argument": WARMUP_ARGUMENT,
    "warmup_fraction": WARMUP_FRACTION,
    "max_steps": MAX_STEPS,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "gpu": torch.cuda.get_device_name(0),
    "compute_capability": torch.cuda.get_device_capability(0),
    "cuda_runtime": torch.version.cuda,
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "peft": peft.__version__,
    "trl": trl.__version__,
    "bitsandbytes": bitsandbytes.__version__,
    "datasets": datasets.__version__,
    "accelerate": accelerate.__version__,
    "safetensors": safetensors.__version__,
    "base_model": BASE_MODEL,
    "base_revision": BASE_REVISION,
    "seed": SEED,
})


## Frozen synthetic fixtures

Precision examples exercise extraction and evidence linking. Writer examples
exercise short recruiter messages, grounded cover-letter language and résumé
bullet revision in German and English. Every desired answer is constrained to
facts stated in the input.


In [ ]:
FIXTURES = [
    {
        "id": "precision-en-01", "slot": "precision", "split": "train",
        "language": "en",
        "system": "Extract only explicit job requirements. Return compact JSON.",
        "user": "Data Analyst. Required: SQL and Power BI. German B2 is preferred. Hybrid in Berlin.",
        "assistant": '{"required_skills":["SQL","Power BI"],"preferred_languages":["German B2"],"work_mode":"hybrid","city":"Berlin"}',
    },
    {
        "id": "precision-de-01", "slot": "precision", "split": "train",
        "language": "de",
        "system": "Extrahiere nur ausdrücklich genannte Anforderungen als kompaktes JSON.",
        "user": "Für die Stelle als Produktionsplaner sind SAP PP und Deutsch C1 erforderlich. Erfahrung mit Lean ist wünschenswert.",
        "assistant": '{"required_skills":["SAP PP"],"required_languages":["Deutsch C1"],"preferred_skills":["Lean"]}',
    },
    {
        "id": "precision-en-02", "slot": "precision", "split": "train",
        "language": "en",
        "system": "Link requirements only to quoted candidate evidence. Use null when unsupported.",
        "user": "Requirement: lifecycle email automation. Evidence: Managed weekly Mailchimp campaigns and audience segments. Requirement: Salesforce Marketing Cloud.",
        "assistant": '{"links":[{"requirement":"lifecycle email automation","evidence":"Managed weekly Mailchimp campaigns and audience segments."},{"requirement":"Salesforce Marketing Cloud","evidence":null}]}',
    },
    {
        "id": "precision-de-02", "slot": "precision", "split": "train",
        "language": "de",
        "system": "Verknüpfe Anforderungen nur mit wörtlich vorhandenen Nachweisen.",
        "user": "Anforderung: Lieferantenmanagement. Nachweis: Koordinierte drei lokale Lieferanten für ein Hochschulprojekt. Anforderung: Six Sigma Zertifikat.",
        "assistant": '{"links":[{"requirement":"Lieferantenmanagement","evidence":"Koordinierte drei lokale Lieferanten für ein Hochschulprojekt."},{"requirement":"Six Sigma Zertifikat","evidence":null}]}',
    },
    {
        "id": "precision-en-03", "slot": "precision", "split": "train",
        "language": "en",
        "system": "Classify explicit constraints without guessing.",
        "user": "The role is a 20-hour working-student contract. Candidates must be enrolled and available on Tuesdays.",
        "assistant": '{"employment":"working_student","hours_per_week":20,"required_enrollment":true,"required_days":["Tuesday"]}',
    },
    {
        "id": "precision-de-03", "slot": "precision", "split": "train",
        "language": "de",
        "system": "Klassifiziere ausdrücklich genannte Bedingungen ohne Vermutungen.",
        "user": "Die Tätigkeit ist vor Ort in Ulm. Ein Führerschein Klasse B ist zwingend. Reisebereitschaft wird nicht erwähnt.",
        "assistant": '{"work_mode":"on_site","city":"Ulm","required_credentials":["Führerschein Klasse B"],"travel_required":null}',
    },
    {
        "id": "precision-en-eval", "slot": "precision", "split": "eval",
        "language": "en",
        "system": "Extract only explicit requirements. Return compact JSON.",
        "user": "CRM Specialist. HubSpot is required; SQL is a plus. Remote within Germany. English C1.",
        "assistant": '{"required_skills":["HubSpot"],"preferred_skills":["SQL"],"work_mode":"remote_germany","required_languages":["English C1"]}',
    },
    {
        "id": "precision-de-eval", "slot": "precision", "split": "eval",
        "language": "de",
        "system": "Extrahiere nur ausdrücklich genannte Anforderungen als kompaktes JSON.",
        "user": "Gesucht wird ein Junior Controller mit Excel. Power BI ist von Vorteil. Arbeitsort ist Köln, zwei Homeoffice-Tage sind möglich.",
        "assistant": '{"seniority":"junior","required_skills":["Excel"],"preferred_skills":["Power BI"],"city":"Köln","remote_days_per_week":2}',
    },
    {
        "id": "writer-en-01", "slot": "writer", "split": "train",
        "language": "en",
        "system": "Write a concise recruiter message using only supplied facts.",
        "user": "Role: Junior Data Analyst at Northstar. Candidate evidence: built Power BI dashboards in a university project; uses SQL in coursework. Ask for a short conversation.",
        "assistant": "Hello Northstar team, I am interested in the Junior Data Analyst role. I built Power BI dashboards for a university project and use SQL in my coursework. Would you be open to a short conversation about the position? Best regards",
    },
    {
        "id": "writer-de-01", "slot": "writer", "split": "train",
        "language": "de",
        "system": "Schreibe eine kurze Recruiter-Nachricht nur mit den gelieferten Fakten.",
        "user": "Rolle: CRM Specialist bei Nordlicht. Nachweis: zwei Jahre Mailchimp-Kampagnen und Zielgruppensegmentierung. Bitte um ein kurzes Gespräch.",
        "assistant": "Guten Tag liebes Nordlicht-Team, die Position als CRM Specialist interessiert mich sehr. Ich bringe zwei Jahre Erfahrung mit Mailchimp-Kampagnen und Zielgruppensegmentierung mit. Hätten Sie Zeit für ein kurzes Gespräch? Freundliche Grüße",
    },
    {
        "id": "writer-en-02", "slot": "writer", "split": "train",
        "language": "en",
        "system": "Revise one résumé bullet. Preserve facts and do not add metrics.",
        "user": "Original: Helped the team with weekly sales reports. Evidence: Prepared weekly Excel reports and checked missing values.",
        "assistant": "Prepared weekly Excel sales reports and checked source data for missing values.",
    },
    {
        "id": "writer-de-02", "slot": "writer", "split": "train",
        "language": "de",
        "system": "Überarbeite einen Lebenslauf-Stichpunkt. Erhalte alle Fakten und erfinde keine Kennzahlen.",
        "user": "Original: Unterstützung bei der Produktionsplanung. Nachweis: Pflege von SAP-PP-Auftragsdaten und Abstimmung mit der Fertigung.",
        "assistant": "Pflegte SAP-PP-Auftragsdaten und stimmte Planungsänderungen mit der Fertigung ab.",
    },
    {
        "id": "writer-en-03", "slot": "writer", "split": "train",
        "language": "en",
        "system": "Write one grounded cover-letter paragraph with a natural tone.",
        "user": "Role: Marketing Analyst. Company: Birch. Evidence: Google Analytics coursework, an A/B testing project, and Excel campaign reports. Do not claim employment experience.",
        "assistant": "I am drawn to Birch's Marketing Analyst role because it combines measurement with practical campaign decisions. Through Google Analytics coursework, an A/B testing project, and Excel campaign reporting, I have built a solid foundation that I would be keen to apply and deepen in your team.",
    },
    {
        "id": "writer-de-03", "slot": "writer", "split": "train",
        "language": "de",
        "system": "Schreibe einen belegbaren Anschreiben-Absatz in natürlichem Ton.",
        "user": "Rolle: Junior Produktionsplaner. Unternehmen: Werkblick. Nachweis: Hochschulprojekt zur Kapazitätsplanung, SAP-PP-Grundkenntnisse. Keine Berufserfahrung behaupten.",
        "assistant": "Die Position als Junior Produktionsplaner bei Werkblick spricht mich an, weil sie analytische Planung mit enger Abstimmung in der Fertigung verbindet. In einem Hochschulprojekt zur Kapazitätsplanung und durch meine SAP-PP-Grundkenntnisse habe ich dafür eine solide Grundlage aufgebaut, die ich in Ihrem Team weiterentwickeln möchte.",
    },
    {
        "id": "writer-en-eval", "slot": "writer", "split": "eval",
        "language": "en",
        "system": "Write a concise recruiter message using only supplied facts.",
        "user": "Role: Working Student BI at Elm. Evidence: Power BI dashboard project and Excel. Ask whether applications are still being reviewed.",
        "assistant": "Hello Elm team, I am interested in the Working Student BI role. I have completed a Power BI dashboard project and work with Excel. Are applications for the role still being reviewed? Best regards",
    },
    {
        "id": "writer-de-eval", "slot": "writer", "split": "eval",
        "language": "de",
        "system": "Überarbeite einen Lebenslauf-Stichpunkt. Erhalte alle Fakten und erfinde keine Kennzahlen.",
        "user": "Original: Newsletter gemacht. Nachweis: Monatliche Newsletter in Mailchimp erstellt und Links vor Versand geprüft.",
        "assistant": "Erstellte monatliche Newsletter in Mailchimp und prüfte sämtliche Links vor dem Versand.",
    },
]

selected = [row for row in FIXTURES if row["slot"] == ADAPTER_SLOT]
train_rows = [row for row in selected if row["split"] == "train"]
eval_rows = [row for row in selected if row["split"] == "eval"]
assert train_rows and eval_rows
assert not ({row["id"] for row in train_rows} & {row["id"] for row in eval_rows})

def canonical_hash(value):
    encoded = json.dumps(
        value, ensure_ascii=False, sort_keys=True, separators=(",", ":")
    ).encode("utf-8")
    return hashlib.sha256(encoded).hexdigest()

DATASET_SHA256 = canonical_hash(selected)
print({
    "train_examples": len(train_rows),
    "held_out_examples": len(eval_rows),
    "dataset_sha256": DATASET_SHA256,
})


In [ ]:
# torch.cuda.is_bf16_supported() takes `including_emulation=True` by default,
# so it answers True on a Tesla T4 (compute capability 7.5) even though the
# card has no bf16 tensor cores. Gating on it sends the run down an emulated
# bf16 path on hardware whose native acceleration is fp16, and it also makes
# the float32 upcast further down this cell look unnecessary when it is not.
# Ask the compute capability instead: bf16 tensor cores arrive with Ampere.
#
# The two dtypes are not symmetric in what they demand of the rest of the run.
# fp16 goes through torch.amp.GradScaler, whose unscale kernel
# `_amp_foreach_non_finite_check_and_unscale_cuda` is implemented for float32
# and float16 gradients only; a single bfloat16 trainable tensor makes the
# first optimizer step raise NotImplementedError. bf16 uses no scaler at all
# and so has no such constraint. The training cell enforces the float32
# invariant that fp16 requires.
#
# PRECISION_MODE is the beginner-visible switch:
#   "bf16"  force bfloat16 (the default). Emulated on a pre-Ampere card and so
#           slower per step, but it uses no GradScaler, so no step is ever
#           skipped and the learning-rate schedule always advances.
#   "auto"  bfloat16 when the GPU has bf16 tensor cores (Ampere, compute
#           capability 8.0 or above), float16 otherwise. Matches the hardware,
#           and is the right choice for a long run.
#   "fp16"  force float16.
#
# The default is "bf16" rather than "auto" because of the step budget, not the
# hardware. fp16 goes through GradScaler, which begins at init_scale=65536 and,
# on overflow, skips the optimizer step and halves the scale. Transformers only
# calls lr_scheduler.step() when the optimizer actually ran, so a skipped step
# freezes the schedule as well. Nine halvings reach 128, which cost a twelve-step
# run ten of its twelve updates and left the loss flat to four decimal places
# until the very end. A thousand-step run absorbs that; this one cannot.
# Choose "auto" only if MAX_STEPS is raised into the hundreds.
PRECISION_MODE = "bf16"
if PRECISION_MODE not in {"auto", "bf16", "fp16"}:
    raise ValueError('PRECISION_MODE must be "auto", "bf16" or "fp16"')

BF16_TENSOR_CORES = torch.cuda.get_device_capability(0) >= (8, 0)
if PRECISION_MODE == "bf16":
    compute_dtype = torch.bfloat16
elif PRECISION_MODE == "fp16":
    compute_dtype = torch.float16
else:
    compute_dtype = torch.bfloat16 if BF16_TENSOR_CORES else torch.float16
print({
    "precision_mode": PRECISION_MODE,
    "compute_capability": torch.cuda.get_device_capability(0),
    "bf16_tensor_cores": BF16_TENSOR_CORES,
    "compute_dtype": str(compute_dtype),
    "is_bf16_supported_reports": torch.cuda.is_bf16_supported(),
})
quantization = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    revision=BASE_REVISION,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Cap every card explicitly. With device_map="auto" and no cap, accelerate can
# fill one T4 to about 11 GiB and leave nothing for the training step. The CPU
# budget is zero so an impossible split fails immediately instead of silently
# offloading to host memory and running for hours.
max_memory = {}
for device_index in range(torch.cuda.device_count()):
    total_gib = torch.cuda.get_device_properties(device_index).total_memory / 2**30
    max_memory[device_index] = f"{max(1, int(total_gib) - 3)}GiB"
max_memory["cpu"] = "0GiB"

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    revision=BASE_REVISION,
    quantization_config=quantization,
    dtype=compute_dtype,
    device_map="auto",
    max_memory=max_memory,
    low_cpu_mem_usage=True,
)

# prepare_model_for_kbit_training is deliberately not called here. It upcasts
# every non-4bit floating-point parameter to float32, and on Qwen3.5-9B the
# embedding and the output projection stay outside the 4-bit quantisation.
# Those two tensors are around a billion parameters each, so the upcast asks a
# 14.56 GiB T4 for several GiB more than it has and the run dies with
# OutOfMemoryError before the first step.
#
# The three things that helper actually does for QLoRA are reproduced below,
# with the float32 upcast limited to the small normalisation layers, which is
# where it buys numerical stability at negligible memory cost.
model.config.use_cache = False
for parameter in model.parameters():
    parameter.requires_grad = False
for module_name, module in model.named_modules():
    if "norm" in module_name.lower():
        module.to(torch.float32)
model.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={"use_reentrant": False},
)
model.enable_input_require_grads()

# hf_device_map is only attached when accelerate actually splits the model
# across devices. A single-GPU placement leaves the attribute unset.
placement = getattr(model, "hf_device_map", None) or {
    "": str(next(model.parameters()).device),
}
print({
    "max_memory": {str(key): value for key, value in max_memory.items()},
    "placement": {str(key): str(value) for key, value in placement.items()},
    "footprint_gib": round(model.get_memory_footprint() / 2**30, 2),
    "reserved_gib": round(torch.cuda.memory_reserved() / 2**30, 2),
})

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)
model = get_peft_model(model, lora_config)

# GradScaler cannot unscale *bfloat16* gradients (float16 is fine), and
# skipping prepare_model_for_kbit_training means the adapters inherit whatever
# the base model's compute dtype is. Upcast only the trainable adapter
# tensors: a few million parameters, against the billions the blanket upcast
# would have touched. This upcast is necessary but not sufficient — the
# training cell re-asserts it, because it does not survive everything the
# trainer does to the model.
for parameter in model.parameters():
    if parameter.requires_grad:
        parameter.data = parameter.data.to(torch.float32)

model.print_trainable_parameters()
print({
    "trainable_dtypes_after_upcast": sorted({
        str(parameter.dtype)
        for parameter in model.parameters()
        if parameter.requires_grad
    }),
})


## Held-out probe

Training loss on six memorised fixtures stops being informative long before the
step budget runs out. This cell defines a callback that measures the held-out
fixtures *during* training and decides what to keep on the basis of what it
measured.

Three signals per row: **held-out loss** (per row, because the aggregate hid
that one row started at 1.3631 and the other at 0.5464), **a degeneracy score**
for repetition loops, and **a language check** for the failure neither of the
others could see — an adapter answering an English prompt with a memorised
German template, fluently, while improving the loss.

The selection rule matters as much as the signals. The lowest-loss step is not
automatically kept: only a step whose guards actually ran *and* passed is ever
snapshotted, and after restoring, the guards run once more on the exact weights
about to be saved. A previous version treated "not measured" as "passed" and
shipped a broken adapter labelled promotable.

The held-out fixtures are still never trained on. The probe restores every RNG
stream it touches, so the training trajectory is identical to what it would have
been without the probe attached.


In [ ]:
import json as _json
import re
import time

# What three probed runs established, and what each one missed.
#
# Precision: held-out loss bottomed at step 10 and rose to step 60 while
# training loss fell to 8.2e-05. The German repetition loop was progressive, not
# a bf16 fault. Early stopping plus best-checkpoint restore cut the run to 15
# steps and shipped step 10.
#
# Writer, first probe: held-out loss improved 56% while the adapter answered an
# English prompt in German, reproducing 23 characters of writer-de-01's target
# verbatim. Teacher-forced loss cannot see that, because forcing the correct
# prefix hides a first-token divergence, and the degeneracy score cannot either,
# because fluent German scores 0.0. That produced the language guard.
#
# Writer, second probe: the guard worked — it flagged steps 10 and 15 — and the
# run still shipped a broken adapter stamped `promotable: true`. The lowest loss
# was step 16, which was not on the every-five generation grid, so no guard ran
# there; `verdict()` read the absent result as a pass. The artifact's own
# post-training generations were German.
#
# The lesson is not "add another metric". It is that a guard which silently
# passes when it did not run is worse than no guard, because it converts an
# unknown into a claim. This version closes that:
#
#   1. Generation runs on every step that sets a new loss minimum, on top of the
#      fixed grid. Any step that could be kept is checked.
#   2. Two bests are tracked. `best_any` is the lowest loss and drives patience.
#      `best_clean` is the lowest loss among steps whose guards ran and passed,
#      and is the only thing ever snapshotted or restored.
#   3. `guardsPassed` is None when the guards did not run — explicitly null, not
#      an absent key, so nothing can mistake it for a pass again.
#   4. After restoring, the guards run once more on the weights actually about
#      to be hashed and zipped. `verdict()` reports that check, not a step from
#      the training history.
#
# On the writer curve of 2026-08-09 this changes what ships. Step 16 had the
# lowest loss and generated German; steps 20 and 24 were English at a higher
# loss; and steps 8 and 9 were never generated at all despite sitting close to
# the minimum. Rule 1 makes those candidates visible, and rule 2 keeps the best
# one that is actually usable.
#
# The probe still must not change the run it measures. It restores training
# mode, the cache flag and every RNG stream it touches, and that is measured,
# not asserted: with the probe attached, the 60-step Precision run reproduced
# train_loss 0.2458519255937669 and total_flos 1576347950979072 exactly.

PROBE_LOSS_EVERY_STEPS = 1
PROBE_GENERATE_EVERY_STEPS = 5
# Precision targets are compact JSON, none over 188 characters. Writer targets
# are prose reaching 332 characters (writer-de-03), and German compounds
# tokenise to more tokens per character than a chars/4 estimate suggests. A cap
# below the target length truncates the generation, which corrupts the
# degeneracy score and the language guard alike, since both would be reading a
# prefix. Measured: the longest writer generation ran 203 characters, so 192
# leaves real headroom.
PROBE_MAX_NEW_TOKENS = 192 if ADAPTER_SLOT == "writer" else 96
# The post-restore check generates at the length the held-out cell uses, so the
# thing being verified is the thing that gets reported, not a shorter proxy.
PROBE_VERIFY_MAX_NEW_TOKENS = 256

# A twelve-character window is long enough that ordinary JSON keys repeating
# across a short object do not register, and short enough to catch a loop
# within the first line of one. The measured Precision loop scores 0.9771
# against 0.00 to 0.07 for every well-formed output, so the threshold sits in a
# gap rather than on a slope. All eight writer targets score 0.0, so the same
# threshold carries to prose with a wider margin, not a narrower one.
PROBE_DEGENERACY_WINDOW = 12
PROBE_DEGENERACY_THRESHOLD = 0.5

# Stop after this many consecutive probes without a new minimum.
#
# The first writer run used patience 5 and min_delta 1e-3 and stopped at step 15
# on a curve that was heading back down: minimum 0.4933 at step 10, up to 0.5200
# at 13, then 0.5090 and 0.4943. It ended 0.001007 above the best and falling at
# 0.0147 per step, having missed the improvement threshold by a thousandth.
# Precision's curve rose monotonically after its minimum and was never at risk
# of this. Prose descends more gently and in steps, so both constants are
# loosened: a tenth of the previous delta, and enough patience to sit through a
# five-step excursion like the one above.
EARLY_STOP_PATIENCE = 8
EARLY_STOP_MIN_DELTA = 1e-4

# Function words only — content words carry the topic between languages and
# would fire on a correct translation of the same subject. The intersection is
# removed below so shared spellings such as "in", "am", "so" and "was" cannot
# vote for either side.
LANGUAGE_MARKERS = {
    "en": {
        "the", "a", "an", "and", "or", "but", "is", "are", "were", "i", "you",
        "he", "she", "it", "we", "they", "with", "for", "of", "to", "at",
        "from", "after", "about", "on", "not", "also", "still", "very", "as",
        "that", "this", "these", "would", "could", "should", "have", "has",
        "had", "can", "my", "your", "their", "our", "am", "been", "being",
        "there", "which", "while", "because", "through", "into", "please",
    },
    "de": {
        "der", "die", "das", "den", "dem", "des", "ein", "eine", "einen",
        "einem", "einer", "und", "oder", "aber", "ist", "sind", "war", "ich",
        "du", "er", "sie", "es", "wir", "ihr", "mit", "für", "von", "zu",
        "bei", "aus", "nach", "über", "auf", "nicht", "auch", "noch", "sehr",
        "als", "wie", "dass", "sowie", "durch", "werden", "wurde", "haben",
        "hat", "hatte", "kann", "am", "im", "so", "was", "sich", "mich",
        "meine", "meiner", "meinen", "ihre", "ihren", "gerne", "bringe",
    },
}
_shared = LANGUAGE_MARKERS["en"] & LANGUAGE_MARKERS["de"]
LANGUAGE_MARKERS = {
    code: words - _shared for code, words in LANGUAGE_MARKERS.items()
}
# Below these, the text is treated as undecidable rather than forced into a
# guess. Compact JSON has almost no function words and must come back None, or
# the guard would fire on every Precision generation.
LANGUAGE_MIN_WORDS = 8
LANGUAGE_MIN_RATE = 0.08
LANGUAGE_MIN_MARGIN = 1.5


def detect_language(text):
    """
    Return "en", "de" or None for text that does not clearly look like either.

    A frequency count over function words, not a real language identifier. It
    exists to catch one specific failure — the adapter answering an English
    prompt with a memorised German template — and it deliberately abstains
    rather than guessing on JSON, code or very short strings.
    """
    words = re.findall(r"[a-zA-ZäöüÄÖÜß]+", text.lower())
    if len(words) < LANGUAGE_MIN_WORDS:
        return None
    rates = {
        code: sum(word in markers for word in words) / len(words)
        for code, markers in LANGUAGE_MARKERS.items()
    }
    best_code = max(rates, key=rates.get)
    other = max(rate for code, rate in rates.items() if code != best_code)
    if rates[best_code] < LANGUAGE_MIN_RATE:
        return None
    if other > 0 and rates[best_code] < other * LANGUAGE_MIN_MARGIN:
        return None
    return best_code


def probe_prompt_text(row):
    """The exact prompt string the held-out cell will use after training."""
    return tokenizer.apply_chat_template(
        [
            {"role": "system", "content": row["system"]},
            {"role": "user", "content": row["user"]},
        ],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )


def probe_full_text(row):
    """
    The prompt with the target answer appended, rendered by the chat template.

    Built through the template rather than by concatenating the answer and an
    EOS token, so the sequence the loss is computed over is the one the trainer
    would have built. The template owns the turn terminator; guessing it as
    tokenizer.eos_token happened to be right for this checkpoint and is not a
    property worth relying on.
    """
    return tokenizer.apply_chat_template(
        [
            {"role": "system", "content": row["system"]},
            {"role": "user", "content": row["user"]},
            {"role": "assistant", "content": row["assistant"]},
        ],
        tokenize=False,
        add_generation_prompt=False,
        enable_thinking=False,
    )


def degeneracy_score(text, window=PROBE_DEGENERACY_WINDOW):
    """
    Fraction of character n-grams in `text` that repeat an earlier one.

    Character n-grams rather than words, because the failure this is built to
    catch — `"employment":"employment":"employment":` — contains no spaces at
    all and would score zero on any whitespace tokenisation. Texts shorter than
    one window cannot be scored and return 0.0; that is a blind spot for a very
    short loop, and it is accepted because a generation that short has already
    failed the length checks that matter.
    """
    if len(text) < window + 1:
        return 0.0
    grams = [text[index:index + window] for index in range(len(text) - window + 1)]
    return round(1.0 - len(set(grams)) / len(grams), 4)


def completion_label_span(row):
    """
    Token ids for prompt+answer, and the index where the answer starts.

    The previous version tokenised `prompt` and `prompt + answer` separately and
    assumed the first len(prompt_ids) tokens of the join equalled the prompt's
    own tokenisation. BPE does not guarantee that: a merge across the boundary
    shifts the split and silently moves the mask, either scoring trailing prompt
    tokens or dropping the answer's first token. Here the prefix relationship is
    checked, and a mismatch falls back to character offsets instead of being
    absorbed. If both routes fail the run stops, because a wrong mask produces a
    plausible number that means nothing.
    """
    prompt_text = probe_prompt_text(row)
    full_text = probe_full_text(row)
    if not full_text.startswith(prompt_text):
        raise RuntimeError({
            "message": (
                "The chat template does not render the prompt as a prefix of "
                "the full conversation, so the completion cannot be masked by "
                "position. Inspect the template before trusting any loss."
            ),
            "row": row["id"],
        })
    full = tokenizer(full_text, return_tensors="pt", add_special_tokens=False)
    full_ids = full["input_ids"]
    prompt_ids = tokenizer(
        prompt_text, return_tensors="pt", add_special_tokens=False
    )["input_ids"]
    boundary = int(prompt_ids.shape[-1])
    clean_prefix = (
        boundary <= int(full_ids.shape[-1])
        and torch.equal(full_ids[0, :boundary], prompt_ids[0])
    )
    if not clean_prefix:
        offsets = tokenizer(
            full_text, add_special_tokens=False, return_offsets_mapping=True
        )["offset_mapping"]
        boundary = next(
            (
                index
                for index, (start, _end) in enumerate(offsets)
                if start >= len(prompt_text)
            ),
            None,
        )
        if boundary is None:
            raise RuntimeError({
                "message": "Could not locate the completion boundary.",
                "row": row["id"],
            })
    if boundary >= int(full_ids.shape[-1]):
        raise RuntimeError({
            "message": "The completion masked away to nothing.",
            "row": row["id"],
        })
    if int(full_ids.shape[-1]) > MAX_LENGTH:
        raise RuntimeError({
            "message": "A held-out row is longer than MAX_LENGTH.",
            "row": row["id"],
            "tokens": int(full_ids.shape[-1]),
            "max_length": MAX_LENGTH,
        })
    return full, boundary, clean_prefix


def row_held_out_loss(row):
    """Completion-only loss for one row, with the number of tokens it scored."""
    encoded, boundary, _clean = completion_label_span(row)
    input_ids = encoded["input_ids"].to(model.device)
    attention_mask = encoded["attention_mask"].to(model.device)
    labels = input_ids.clone()
    labels[:, :boundary] = -100
    with torch.inference_mode():
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )
    scored_tokens = int((labels[:, 1:] != -100).sum())
    return float(outputs.loss), scored_tokens


def held_out_loss(rows):
    """
    Token-weighted completion-only loss, plus the per-row breakdown.

    The aggregate alone is misleading whenever the rows differ in difficulty.
    On the writer slot the two held-out rows start at 1.3980 and 0.5752, and
    reading only the mean hid that the improvement was concentrated in the row
    whose generation had switched language.
    """
    per_row = {}
    total_loss = 0.0
    total_tokens = 0
    for row in rows:
        loss, scored_tokens = row_held_out_loss(row)
        per_row[row["id"]] = round(loss, 6)
        total_loss += loss * scored_tokens
        total_tokens += scored_tokens
    return total_loss / max(1, total_tokens), per_row


def probe_generate(row, max_new_tokens):
    """Greedy generation for one fixture under whatever adapter state is active."""
    encoded = tokenizer(
        [probe_prompt_text(row)],
        return_tensors="pt",
        padding=True,
    ).to(model.device)
    with torch.inference_mode():
        generated = model.generate(
            **encoded,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    new_tokens = generated[0, encoded["input_ids"].shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


# Language detection abstains on compact JSON — too few function words to
# classify — which is correct, and it leaves the Precision slot with only the
# degeneracy guard. Degeneracy catches the `"employment":"employment":` loop but
# not a target that came back as prose, a truncated object or a stray code
# fence. The structural check below is the Precision-side equivalent of the
# language check: it asks whether the output is even the right *kind* of thing.
#
# It is driven by the data rather than by ADAPTER_SLOT. A fixture whose target
# parses as JSON demands a generation that parses as JSON; a prose fixture is
# exempt automatically. Nothing needs configuring per slot, and a future mixed
# slot behaves correctly without a new flag.
CODE_FENCE = re.compile(r"^\s*```(?:json)?\s*(.*?)\s*```\s*$", re.DOTALL)


def strip_code_fence(text):
    """Return the text inside a markdown fence, and whether one was present."""
    match = CODE_FENCE.match(text)
    return (match.group(1), True) if match else (text, False)


def parses_as_json(text):
    try:
        _json.loads(text)
    except Exception:
        return False
    return True


def structural_check(row, text):
    """
    Whether `text` is the same kind of object as the fixture's target.

    A fence is unwrapped before parsing rather than failed outright: emitting
    fenced JSON is a formatting deviation worth recording, not a broken output,
    and treating it as a guard failure would reject checkpoints that are merely
    untidy. It is reported as usedCodeFence so it stays visible.
    """
    expects_json = parses_as_json(row["assistant"])
    if not expects_json:
        return {"expectsJson": False, "jsonInvalid": False, "usedCodeFence": False}
    unwrapped, fenced = strip_code_fence(text)
    return {
        "expectsJson": True,
        "jsonInvalid": not parses_as_json(unwrapped),
        "usedCodeFence": fenced,
    }


def score_generation(row, text):
    """Everything measurable about one generation, without a GPU."""
    detected = detect_language(text)
    structural = structural_check(row, text)
    return {
        "id": row["id"],
        "language": row["language"],
        "generated": text,
        "degeneracyScore": degeneracy_score(text),
        "expectsJson": structural["expectsJson"],
        "jsonInvalid": structural["jsonInvalid"],
        "usedCodeFence": structural["usedCodeFence"],
        "languageDetected": detected,
        # None means undecidable, not wrong: compact JSON has too few function
        # words to classify and must not trip the guard on the Precision slot.
        "languageMismatch": detected is not None and detected != row["language"],
        # Exact string equality. Informative for the precision slot, where the
        # target is one canonical JSON object, and meaningless for the writer
        # slot, where a thousand phrasings could be correct. On writer runs read
        # heldOutLoss, languageMismatch and degeneracyScore instead; False here
        # is the expected value, not a finding.
        "matchesExpected": text == row["assistant"],
    }


class HeldOutProbe(TrainerCallback):
    """
    Measures the held-out fixtures during training, keeps the best checkpoint
    that passes every guard, and stops once the loss stops improving.

    The distinction between "best" and "best that passed" is the whole point.
    On the writer run of 2026-08-09 the lowest held-out loss was step 16, which
    was not on the generation grid, so no guard ever ran there. The previous
    version read the absent result as a pass, restored step 16, and wrote
    `promotable: true` into an artifact whose English fixture answered in
    German. Two rules follow, and they are enforced below rather than
    documented as advice:

      * A step is only a candidate if its guards were actually evaluated.
      * The snapshot tracks the best *guard-passing* step, not the best loss.

    To make the first rule cheap, generation now runs on every step that sets a
    new loss minimum, in addition to the fixed grid. Candidates are exactly the
    steps worth generating for, so nothing that could be kept goes unchecked.
    """

    def __init__(
        self,
        rows,
        loss_every_steps,
        generate_every_steps,
        max_new_tokens,
        patience,
        min_delta,
        verify_max_new_tokens,
    ):
        self.rows = list(rows)
        self.loss_every_steps = loss_every_steps
        self.generate_every_steps = generate_every_steps
        self.max_new_tokens = max_new_tokens
        self.patience = patience
        self.min_delta = min_delta
        self.verify_max_new_tokens = verify_max_new_tokens
        self.history = []
        self.base_control = []
        self.seconds_spent = 0.0
        # best_any is the lowest loss seen at all, and drives the patience
        # counter: while the loss is still falling the run has not converged,
        # whether or not the current region generates usable text.
        self.best_any = None
        # best_clean is the lowest loss among steps whose guards ran and
        # passed. This is the only thing ever snapshotted or restored.
        self.best_clean = None
        self.best_state = None
        self.probes_since_best = 0
        self.should_stop = False
        self.stopped_at_step = None
        self.restored_from_step = None
        self.restored_selection = None
        self.post_restore_check = None

    # -- model state helpers -------------------------------------------------

    def _enter_eval(self):
        was_training = model.training
        cache_flag = model.config.use_cache
        rng = {
            "cpu": torch.get_rng_state(),
            "cuda": torch.cuda.get_rng_state_all(),
            "python": random.getstate(),
            "numpy": np.random.get_state(),
        }
        model.eval()
        model.config.use_cache = True
        return was_training, cache_flag, rng

    def _exit_eval(self, was_training, cache_flag, rng):
        model.config.use_cache = cache_flag
        if was_training:
            model.train()
        torch.set_rng_state(rng["cpu"])
        torch.cuda.set_rng_state_all(rng["cuda"])
        random.setstate(rng["python"])
        np.random.set_state(rng["numpy"])

    def _snapshot_trainables(self):
        """Copy the trainable tensors to host memory, detached from the graph."""
        return {
            name: parameter.detach().to("cpu", copy=True)
            for name, parameter in model.named_parameters()
            if parameter.requires_grad
        }

    def _generate_and_score(self, max_new_tokens):
        return [
            score_generation(row, probe_generate(row, max_new_tokens))
            for row in self.rows
        ]

    @staticmethod
    def _guard_failures(generations):
        """Ids failing any guard. Empty means every guard passed on every row."""
        return sorted({
            item["id"]
            for item in generations
            if item["languageMismatch"]
            or item["jsonInvalid"]
            or item["degeneracyScore"] >= PROBE_DEGENERACY_THRESHOLD
        })

    # -- measurement ---------------------------------------------------------

    def _record(self, step, scheduled_generate):
        started = time.time()
        was_training, cache_flag, rng = self._enter_eval()
        try:
            aggregate, per_row = held_out_loss(self.rows)
            improved_any = (
                self.best_any is None
                or aggregate < self.best_any["heldOutLoss"] - self.min_delta
            )
            # Generate whenever this step could be kept. Two ways that happens.
            # A new global minimum is a candidate by definition — that is the
            # case step 16 fell into, between grid points and never checked. But
            # so is any step that beats the best guard-passing loss so far, even
            # when it does not beat the global minimum: after the loss dips into
            # a region that fails the guards, the steps that climb back out are
            # exactly the ones worth keeping, and none of them sets a new
            # minimum. Checking only global minima would miss all of them.
            improves_clean = (
                self.best_clean is None
                or aggregate < self.best_clean["heldOutLoss"] - self.min_delta
            )
            do_generate = scheduled_generate or improved_any or improves_clean
            entry = {
                "step": step,
                "heldOutLoss": aggregate,
                "perRowHeldOutLoss": per_row,
                "guardsEvaluated": do_generate,
            }
            failures = None
            if do_generate:
                generations = self._generate_and_score(self.max_new_tokens)
                failures = self._guard_failures(generations)
                entry["generations"] = generations
                entry["maxDegeneracyScore"] = max(
                    item["degeneracyScore"] for item in generations
                )
                entry["languageMismatchIds"] = [
                    item["id"] for item in generations if item["languageMismatch"]
                ]
                entry["jsonInvalidIds"] = [
                    item["id"] for item in generations if item["jsonInvalid"]
                ]
                entry["guardFailureIds"] = failures
                entry["guardsPassed"] = not failures
            else:
                # Explicitly null rather than absent. An absent key is what the
                # previous verdict() mistook for a pass.
                entry["guardsPassed"] = None

            if improved_any:
                self.best_any = {"step": step, "heldOutLoss": aggregate}
                self.probes_since_best = 0
            else:
                self.probes_since_best += 1
                if self.patience > 0 and self.probes_since_best >= self.patience:
                    self.should_stop = True

            is_best_clean = (
                entry["guardsPassed"] is True
                and (
                    self.best_clean is None
                    or aggregate < self.best_clean["heldOutLoss"] - self.min_delta
                )
            )
            if is_best_clean:
                self.best_clean = {
                    "step": step,
                    "heldOutLoss": aggregate,
                    "perRowHeldOutLoss": per_row,
                }
                self.best_state = self._snapshot_trainables()
            entry["isBestAny"] = improved_any
            entry["isBestClean"] = is_best_clean
            entry["probesSinceBest"] = self.probes_since_best
        finally:
            self._exit_eval(was_training, cache_flag, rng)
        entry["probeSeconds"] = round(time.time() - started, 2)
        self.seconds_spent += entry["probeSeconds"]
        self.history.append(entry)
        summary = {
            "step": step,
            "held_out_loss": round(aggregate, 6),
            "per_row": per_row,
            "guards": (
                "not run" if entry["guardsPassed"] is None
                else ("pass" if entry["guardsPassed"] else f"FAIL {failures}")
            ),
            "best_clean_step": (
                self.best_clean["step"] if self.best_clean else None
            ),
        }
        if is_best_clean:
            summary["new_best_clean"] = True
        print(summary)

    def _fill_guards_on_last(self):
        """
        Attach a sample to the final history entry when the stop lands on a step
        whose guards did not run.

        Calling _record again would append a second entry for the same step and
        advance the patience counter twice.
        """
        entry = self.history[-1]
        if entry.get("guardsEvaluated"):
            return
        was_training, cache_flag, rng = self._enter_eval()
        try:
            generations = self._generate_and_score(self.max_new_tokens)
        finally:
            self._exit_eval(was_training, cache_flag, rng)
        failures = self._guard_failures(generations)
        entry["generations"] = generations
        entry["maxDegeneracyScore"] = max(
            item["degeneracyScore"] for item in generations
        )
        entry["languageMismatchIds"] = [
            item["id"] for item in generations if item["languageMismatch"]
        ]
        entry["jsonInvalidIds"] = [
            item["id"] for item in generations if item["jsonInvalid"]
        ]
        entry["guardFailureIds"] = failures
        entry["guardsEvaluated"] = True
        entry["guardsPassed"] = not failures

    # -- restore and verify --------------------------------------------------

    def _verify_restored(self):
        """
        Re-run both guards on the model as it now stands, at the length the
        held-out cell will use.

        This is the check that answers the only question that matters: not
        whether some step passed during training, but whether the weights about
        to be hashed, zipped and shipped still do.
        """
        was_training, cache_flag, rng = self._enter_eval()
        try:
            generations = self._generate_and_score(self.verify_max_new_tokens)
        finally:
            self._exit_eval(was_training, cache_flag, rng)
        failures = self._guard_failures(generations)
        self.post_restore_check = {
            "maxNewTokens": self.verify_max_new_tokens,
            "generations": generations,
            "guardFailureIds": failures,
            "passed": not failures,
        }
        return self.post_restore_check

    def restore_best(self):
        """
        Write the best guard-passing snapshot back into the live model, then
        verify what was written.

        Everything downstream — the held-out cell, save_pretrained, the hashes
        in provenance.json and the ZIP — reads the model as it stands after this
        call.
        """
        if self.best_state is None:
            self.restored_selection = "none"
            print({
                "restore_best": (
                    "NO CHECKPOINT PASSED THE GUARDS. The model is left as "
                    "trained and must not be promoted. Inspect "
                    "heldOutProbe.history: every step either failed a guard or "
                    "was never evaluated."
                ),
            })
        else:
            with torch.no_grad():
                for name, parameter in model.named_parameters():
                    if name in self.best_state:
                        parameter.data.copy_(
                            self.best_state[name].to(
                                parameter.device, parameter.dtype
                            )
                        )
            self.restored_from_step = self.best_clean["step"]
            self.restored_selection = "best_clean"
            print({
                "restored_from_step": self.best_clean["step"],
                "restored_held_out_loss": round(
                    self.best_clean["heldOutLoss"], 6
                ),
                "lowest_loss_step_overall": self.best_any["step"],
                "lowest_loss_overall": round(self.best_any["heldOutLoss"], 6),
                "note": (
                    "restored the best guard-passing step, which is not the "
                    "lowest-loss step"
                    if self.best_any["step"] != self.best_clean["step"]
                    else "the lowest-loss step also passed the guards"
                ),
            })
        check = self._verify_restored()
        print({
            "post_restore_check": "pass" if check["passed"] else "FAIL",
            "post_restore_failures": check["guardFailureIds"],
            "post_restore_languages": {
                item["id"]: item["languageDetected"]
                for item in check["generations"]
            },
        })
        if not check["passed"]:
            print(
                "DO NOT PROMOTE: the restored weights fail a guard on "
                f"{check['guardFailureIds']}. The artifact will still be "
                "written, with promotable=false recorded in provenance.json."
            )
        return self.best_clean

    # -- callback hooks ------------------------------------------------------

    def on_train_begin(self, args, state, control, **kwargs):
        was_training, cache_flag, rng = self._enter_eval()
        try:
            # disable_adapter() zeroes the LoRA contribution without unloading
            # the 4-bit base, so the control sees the same weights, prompts and
            # decoding settings the adapter will.
            with model.disable_adapter():
                self.base_control = []
                for row in self.rows:
                    loss, _tokens = row_held_out_loss(row)
                    scored = score_generation(
                        row, probe_generate(row, self.max_new_tokens)
                    )
                    scored["heldOutLoss"] = loss
                    self.base_control.append(scored)
        finally:
            self._exit_eval(was_training, cache_flag, rng)
        print({
            "base_control_held_out_loss": {
                item["id"]: round(item["heldOutLoss"], 6)
                for item in self.base_control
            },
            "base_control_language": {
                item["id"]: item["languageDetected"] for item in self.base_control
            },
        })
        self._record(step=0, scheduled_generate=True)

    def on_step_end(self, args, state, control, **kwargs):
        step = int(state.global_step)
        is_last = step >= int(args.max_steps) > 0
        scheduled_generate = is_last or (
            self.generate_every_steps > 0
            and step % self.generate_every_steps == 0
        )
        do_loss = is_last or scheduled_generate or (
            self.loss_every_steps > 0 and step % self.loss_every_steps == 0
        )
        if do_loss:
            self._record(step=step, scheduled_generate=scheduled_generate)
        if self.should_stop and not control.should_training_stop:
            self._fill_guards_on_last()
            self.stopped_at_step = step
            control.should_training_stop = True
            print({
                "early_stop": True,
                "stopped_at_step": step,
                "lowest_loss_step": self.best_any["step"],
                "best_guard_passing_step": (
                    self.best_clean["step"] if self.best_clean else None
                ),
                "probes_without_improvement": self.probes_since_best,
                "patience": self.patience,
                "steps_not_run": max(0, int(args.max_steps) - step),
            })
        return control

    # -- reporting -----------------------------------------------------------

    def best_step(self):
        """Lowest held-out loss, guards notwithstanding. For reporting only."""
        if not self.history:
            return None
        best = min(self.history, key=lambda entry: entry["heldOutLoss"])
        return {
            "step": best["step"],
            "heldOutLoss": best["heldOutLoss"],
            "perRowHeldOutLoss": best["perRowHeldOutLoss"],
            "guardsEvaluated": best["guardsEvaluated"],
            "guardsPassed": best["guardsPassed"],
        }

    def language_mismatch_steps(self):
        return [
            entry["step"]
            for entry in self.history
            if entry.get("languageMismatchIds")
        ]

    def json_invalid_steps(self):
        """Steps whose generations failed to parse as the JSON they promised."""
        return [
            entry["step"]
            for entry in self.history
            if entry.get("jsonInvalidIds")
        ]

    def unevaluated_steps(self):
        """Steps with a loss but no guard result. None of these can be kept."""
        return [
            entry["step"]
            for entry in self.history
            if not entry.get("guardsEvaluated")
        ]

    def verdict(self):
        """
        Whether the weights that were saved are fit to promote.

        Reads the post-restore check, not the training history: the question is
        about the artifact, not about a step that happened along the way.
        """
        if not self.history:
            return {"promotable": False, "reason": "no measurements"}
        lowest = self.best_step()
        check = self.post_restore_check
        base = {item["id"]: item["heldOutLoss"] for item in self.base_control}
        if check is None:
            return {
                "promotable": False,
                "reason": (
                    "restore_best() was never called, so the saved weights "
                    "were never verified"
                ),
                "lowestLossStep": lowest,
            }
        promotable = self.best_clean is not None and check["passed"]
        if self.best_clean is None:
            reason = "no step passed both guards; nothing was safe to restore"
        elif not check["passed"]:
            reason = (
                "the restored weights failed a guard on "
                f"{', '.join(check['guardFailureIds'])}"
            )
        else:
            reason = (
                f"restored step {self.best_clean['step']}, which passed both "
                "guards during training and again after restore"
            )
        return {
            "promotable": promotable,
            "reason": reason,
            "restoredFromStep": self.restored_from_step,
            "restoredSelection": self.restored_selection,
            "restoredHeldOutLoss": (
                round(self.best_clean["heldOutLoss"], 6)
                if self.best_clean
                else None
            ),
            "restoredPerRowHeldOutLoss": (
                self.best_clean["perRowHeldOutLoss"] if self.best_clean else None
            ),
            "lowestLossStep": lowest,
            "lossPaidForPassingGuards": (
                round(
                    self.best_clean["heldOutLoss"] - lowest["heldOutLoss"], 6
                )
                if self.best_clean
                else None
            ),
            "baseModelPerRowLoss": {k: round(v, 6) for k, v in base.items()},
            "postRestoreCheck": {
                "passed": check["passed"],
                "guardFailureIds": check["guardFailureIds"],
                "languages": {
                    item["id"]: item["languageDetected"]
                    for item in check["generations"]
                },
            },
            "languageMismatchSteps": self.language_mismatch_steps(),
            "jsonInvalidSteps": self.json_invalid_steps(),
            "stepsWithoutGuardResult": self.unevaluated_steps(),
        }


print({
    "slot": ADAPTER_SLOT,
    "probe_loss_every_steps": PROBE_LOSS_EVERY_STEPS,
    "probe_generate_every_steps": PROBE_GENERATE_EVERY_STEPS,
    "probe_generates_on_every_new_best": True,
    "probe_max_new_tokens": PROBE_MAX_NEW_TOKENS,
    "probe_verify_max_new_tokens": PROBE_VERIFY_MAX_NEW_TOKENS,
    "early_stop_patience": EARLY_STOP_PATIENCE,
    "early_stop_min_delta": EARLY_STOP_MIN_DELTA,
    "language_markers": {
        code: len(words) for code, words in LANGUAGE_MARKERS.items()
    },
    "probe_rows": [(row["id"], row["language"]) for row in eval_rows],
    "rows_requiring_valid_json": [
        row["id"] for row in eval_rows if parses_as_json(row["assistant"])
    ],
    "max_steps_ceiling": MAX_STEPS,
})


In [ ]:
def training_record(example):
    return {
        "prompt": [
            {"role": "system", "content": example["system"]},
            {"role": "user", "content": example["user"]},
        ],
        "completion": [
            {"role": "assistant", "content": example["assistant"]},
        ],
        "chat_template_kwargs": {"enable_thinking": False},
    }

train_dataset = Dataset.from_list([training_record(row) for row in train_rows])

# Transformers derives updates per epoch as
# len(dataloader) // gradient_accumulation_steps. A remainder there leaves some
# fixtures in a ragged tail that contributes unevenly, and with six fixtures the
# previous value of 4 gave 6 // 4 = 1 update per epoch. Requiring exact division
# makes every optimizer step one full-batch pass over the whole slot, which also
# makes dataset order irrelevant to the gradient.
if len(train_dataset) % GRADIENT_ACCUMULATION_STEPS != 0:
    raise RuntimeError({
        "message": "gradient_accumulation_steps must divide the training set exactly",
        "train_examples": len(train_dataset),
        "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    })

# Build the configuration as an explicit mapping so every keyword can be
# checked against the installed signature before construction. WARMUP_ARGUMENT
# was resolved in the setup cell: "warmup_ratio" on Transformers v4,
# "warmup_steps" on v5, which removed `warmup_ratio` and reads a float below 1
# as a fraction of the total training steps.
training_kwargs = {
    "output_dir": str(OUTPUT_DIR / "checkpoints"),
    "max_length": MAX_LENGTH,
    "packing": False,
    "max_steps": MAX_STEPS,
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "learning_rate": 2e-4,
    WARMUP_ARGUMENT: WARMUP_FRACTION,
    "lr_scheduler_type": "cosine",
    "optim": "paged_adamw_8bit",
    "logging_steps": 1,
    "save_strategy": "no",
    "eval_strategy": "no",
    "report_to": "none",
    "gradient_checkpointing": True,
    "completion_only_loss": True,
    "full_determinism": True,
    "bf16": compute_dtype == torch.bfloat16,
    "fp16": compute_dtype == torch.float16,
    "seed": SEED,
    "data_seed": SEED,
}

unsupported_kwargs = sorted(
    set(training_kwargs) - set(inspect.signature(SFTConfig).parameters)
)
if unsupported_kwargs:
    raise RuntimeError({
        "unsupported_sft_config_parameters": unsupported_kwargs,
        "transformers": transformers.__version__,
        "trl": trl.__version__,
    })

training_args = SFTConfig(**training_kwargs)

# Record what the installed stack resolved the warmup schedule to, so the
# printed value can be checked against the provenance file afterwards.
print({
    "warmup_argument": WARMUP_ARGUMENT,
    "warmup_value_passed": WARMUP_FRACTION,
    "resolved_warmup_steps": getattr(training_args, "warmup_steps", None),
    "resolved_warmup_ratio": getattr(training_args, "warmup_ratio", None),
})

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)

def enforce_float32_trainables(module):
    """Move every trainable tensor to float32 and report what had to move."""
    recast = {}
    for parameter_name, parameter in module.named_parameters():
        if parameter.requires_grad and parameter.dtype != torch.float32:
            recast.setdefault(str(parameter.dtype), []).append(parameter_name)
            # Assigning to .data keeps the Parameter object itself, so an
            # optimizer already holding a reference to it stays valid.
            parameter.data = parameter.data.to(torch.float32)
    return recast


def trainable_dtypes_of(module):
    return sorted({
        str(parameter.dtype)
        for parameter in module.parameters()
        if parameter.requires_grad
    })


class EnforceFloat32Trainables(TrainerCallback):
    """
    fp16 mixed precision unscales gradients through torch.amp.GradScaler, whose
    kernel `_amp_foreach_non_finite_check_and_unscale_cuda` is implemented for
    float32 and float16 only. One bfloat16 trainable tensor therefore makes the
    first optimizer step raise NotImplementedError, several frames deep inside
    accelerate, with a message that says nothing about dtypes being the cause.

    The float32 upcast in the model cell does not reliably survive to the
    training loop, so re-assert it here. on_train_begin runs after accelerate
    has prepared the model and before the first backward pass, which is the
    last point where this can still be corrected.
    """

    def __init__(self):
        self.recast_report = {}
        self.final_dtypes = []

    def on_train_begin(self, args, state, control, model=None, **kwargs):
        target = model if model is not None else trainer.model
        self.recast_report = enforce_float32_trainables(target)
        self.final_dtypes = trainable_dtypes_of(target)
        print({
            "on_train_begin_recast": {
                dtype: len(names) for dtype, names in self.recast_report.items()
            },
            "trainable_dtypes": self.final_dtypes,
        })
        if args.fp16 and self.final_dtypes not in ([], ["torch.float32"]):
            raise RuntimeError({
                "message": (
                    "fp16 mixed precision cannot unscale these gradient dtypes. "
                    'Set PRECISION_MODE = "bf16" in the model cell, restart the '
                    "session and run again."
                ),
                "trainable_dtypes": self.final_dtypes,
            })


float32_enforcer = EnforceFloat32Trainables()
trainer.add_callback(float32_enforcer)

# The probe is registered after the dtype enforcer so that, at on_train_begin,
# the trainable tensors are already float32 before anything generates with them.
held_out_probe = HeldOutProbe(
    rows=eval_rows,
    loss_every_steps=PROBE_LOSS_EVERY_STEPS,
    generate_every_steps=PROBE_GENERATE_EVERY_STEPS,
    max_new_tokens=PROBE_MAX_NEW_TOKENS,
    patience=EARLY_STOP_PATIENCE,
    min_delta=EARLY_STOP_MIN_DELTA,
    verify_max_new_tokens=PROBE_VERIFY_MAX_NEW_TOKENS,
)
trainer.add_callback(held_out_probe)

# Also enforce it now, so the printout shows whether the adapters were already
# out of float32 at trainer-construction time or only later.
pre_train_recast = enforce_float32_trainables(trainer.model)
print({
    "pre_train_recast": {
        dtype: len(names) for dtype, names in pre_train_recast.items()
    },
    "example_recast_parameters": sorted(
        name for names in pre_train_recast.values() for name in names
    )[:5],
    "trainable_dtypes": trainable_dtypes_of(trainer.model),
})

# full_determinism asks PyTorch to refuse every nondeterministic kernel. Some
# attention and checkpointing paths have no deterministic CUDA implementation,
# and a hard refusal would abort a twelve-step feasibility run for no benefit.
# Determinism is still requested; a fallback now prints a warning instead, and
# the notebook already declines to claim bit-for-bit reproducibility across
# Kaggle sessions.
torch.use_deterministic_algorithms(True, warn_only=True)

train_result = trainer.train()
print(train_result.metrics)

# Put the best-scoring adapter back into the live model. Everything downstream —
# the held-out cell, save_pretrained, the hashes in provenance.json and the ZIP —
# reads the model as it stands after this call, so without it the artifact would
# describe whichever step training happened to stop on rather than the one that
# generalised. This is the difference between shipping step 15 and shipping
# step 10.
# Restores the best guard-passing checkpoint — not necessarily the lowest-loss
# one — and then re-runs both guards on the weights it just wrote, at the length
# the held-out cell uses. Everything downstream reads the model as it stands
# after this call, so this is the last point at which a broken adapter can be
# caught before it is hashed and zipped.
held_out_probe.restore_best()

# `promotable` reflects the post-restore check on the saved weights, not a step
# that happened to look good during training. It is false when no step passed
# the guards, and false when the restored weights fail them.
verdict = held_out_probe.verdict()
print(verdict)
if not verdict["promotable"]:
    print(
        "This artifact is NOT promotable: "
        + verdict["reason"]
        + ". It is still written, with the verdict recorded in provenance.json."
    )

print({
    "max_steps_ceiling": MAX_STEPS,
    "stopped_at_step": held_out_probe.stopped_at_step or int(trainer.state.global_step),
    "early_stopped": held_out_probe.stopped_at_step is not None,
    "best_held_out_step": held_out_probe.best_step(),
    "restored_from_step": held_out_probe.restored_from_step,
    "probe_overhead_seconds": round(held_out_probe.seconds_spent, 1),
    "degenerate_probe_steps": [
        entry["step"]
        for entry in held_out_probe.history
        if entry.get("maxDegeneracyScore", 0.0) >= PROBE_DEGENERACY_THRESHOLD
    ],
    "language_mismatch_steps": held_out_probe.language_mismatch_steps(),
    "json_invalid_steps": held_out_probe.json_invalid_steps(),
    "steps_without_guard_result": held_out_probe.unevaluated_steps(),
    "lowest_loss_step": held_out_probe.best_step(),
    "restored_from_step": held_out_probe.restored_from_step,
})


## Held-out smoke check

The following cell generates against the two excluded fixtures twice: once with
the trained adapter active and once with it switched off, under identical greedy
decoding. The second pass is the base-model control. Without it nothing in the
run attributes any part of an output to the twelve training steps, because the
base model alone may already produce the same text.

It records raw outputs for human and deterministic schema/fidelity review; it
does not turn string similarity into a product-quality claim, and a difference
from the base model is evidence that training changed something, not evidence
that it changed something for the better.


In [ ]:
# Gradient checkpointing and the key/value cache are mutually exclusive.
# Leaving it on makes transformers silently disable the cache and generate far
# more slowly, so switch it off now that training is finished.
model.gradient_checkpointing_disable()
model.config.use_cache = True
model.eval()


def generate_for(row):
    """Greedy generation for one fixture, using whatever adapter state is active."""
    prompt = tokenizer.apply_chat_template(
        [
            {"role": "system", "content": row["system"]},
            {"role": "user", "content": row["user"]},
        ],
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = tokenizer(
        [prompt],
        return_tensors="pt",
        padding=True,
    ).to(model.device)
    with torch.inference_mode():
        generated = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    new_tokens = generated[0, inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


held_out_results = []
for row in eval_rows:
    adapter_output = generate_for(row)
    # disable_adapter() zeroes the LoRA contribution for the duration of the
    # block without unloading or reloading the 4-bit base, so the control runs
    # against exactly the same weights, prompt and decoding settings.
    with model.disable_adapter():
        base_output = generate_for(row)
    held_out_results.append({
        "id": row["id"],
        "expected": row["assistant"],
        "generated": adapter_output,
        "baseModelControl": base_output,
        "adapterChangedOutput": adapter_output != base_output,
    })

print(json.dumps(held_out_results, ensure_ascii=False, indent=2))
print({
    "held_out_examples": len(held_out_results),
    "outputs_differing_from_base": sum(
        row["adapterChangedOutput"] for row in held_out_results
    ),
})


In [ ]:
adapter_dir = OUTPUT_DIR / "adapter"
adapter_dir.mkdir(parents=True, exist_ok=True)
model.save_pretrained(adapter_dir, safe_serialization=True)
tokenizer.save_pretrained(adapter_dir)

# PEFT holds `target_modules` as a set and serialises it in whatever order that
# set iterates, so two runs of an identical configuration write
# adapter_config.json files with different sha256 values. Rewrite the field in
# sorted order, and the whole file with sorted keys, before anything is hashed,
# so config hashes can be compared across runs and across slots. PEFT reads the
# field back into a set on load, so ordering does not affect behaviour.
adapter_config_path = adapter_dir / "adapter_config.json"
with open(adapter_config_path, encoding="utf-8") as handle:
    adapter_config = json.load(handle)
if isinstance(adapter_config.get("target_modules"), list):
    adapter_config["target_modules"] = sorted(adapter_config["target_modules"])
with open(adapter_config_path, "w", encoding="utf-8") as handle:
    json.dump(adapter_config, handle, ensure_ascii=False, indent=2, sort_keys=True)
    handle.write("\n")

def file_sha256(file_path):
    digest = hashlib.sha256()
    with open(file_path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

artifact_files = []
for file_path in sorted(path for path in adapter_dir.rglob("*") if path.is_file()):
    artifact_files.append({
        "path": file_path.relative_to(OUTPUT_DIR).as_posix(),
        "bytes": file_path.stat().st_size,
        "sha256": file_sha256(file_path),
    })

provenance = {
    "schemaVersion": 2,
    "promotable": verdict["promotable"],
    "promotableReason": verdict["reason"],
    "experiment": "klar-qwen35-adapter-feasibility-v1",
    "adapterSlot": ADAPTER_SLOT,
    "baseModel": BASE_MODEL,
    "baseRevision": BASE_REVISION,
    "dependencyRevisions": DEPENDENCY_REVISIONS,
    "seed": SEED,
    "determinism": "full_determinism requested; nondeterministic CUDA kernels warn instead of raising",
    "datasetSha256": DATASET_SHA256,
    "trainExampleIds": [row["id"] for row in train_rows],
    "heldOutExampleIds": [row["id"] for row in eval_rows],
    "maxSteps": MAX_STEPS,
    "maxLength": MAX_LENGTH,
    "gradientAccumulationSteps": GRADIENT_ACCUMULATION_STEPS,
    "updatesPerEpoch": len(train_dataset) // GRADIENT_ACCUMULATION_STEPS,
    "warmup": {
        "argument": WARMUP_ARGUMENT,
        "fraction": WARMUP_FRACTION,
        "resolvedWarmupSteps": getattr(training_args, "warmup_steps", None),
        "resolvedWarmupRatio": getattr(training_args, "warmup_ratio", None),
    },
    # The final loss and the per-step log history are the only quantitative
    # record this run produces. Printing them is not enough: they have to live
    # in the artifact for the feasibility claim to be checkable afterwards.
    "float32TrainableEnforcement": {
        "preTrain": {
            dtype: len(names) for dtype, names in pre_train_recast.items()
        },
        "onTrainBegin": {
            dtype: len(names)
            for dtype, names in float32_enforcer.recast_report.items()
        },
        "trainableDtypesAtTrainBegin": float32_enforcer.final_dtypes,
    },
    "trainMetrics": dict(train_result.metrics),
    "trainLogHistory": [dict(entry) for entry in trainer.state.log_history],
    # The probe series is the part of this artifact that can distinguish
    # memorisation from a numerical fault, so it is recorded in full rather
    # than summarised. `bestHeldOutStep` is the step MAX_STEPS should match.
    "heldOutProbe": {
        "lossEverySteps": PROBE_LOSS_EVERY_STEPS,
        "generateEverySteps": PROBE_GENERATE_EVERY_STEPS,
        "maxNewTokens": PROBE_MAX_NEW_TOKENS,
        "degeneracyWindow": PROBE_DEGENERACY_WINDOW,
        "degeneracyThreshold": PROBE_DEGENERACY_THRESHOLD,
        "earlyStopPatience": EARLY_STOP_PATIENCE,
        "earlyStopMinDelta": EARLY_STOP_MIN_DELTA,
        "earlyStopped": held_out_probe.stopped_at_step is not None,
        "stoppedAtStep": held_out_probe.stopped_at_step
        or int(trainer.state.global_step),
        "stepsNotRun": max(
            0,
            MAX_STEPS - (held_out_probe.stopped_at_step
                         or int(trainer.state.global_step)),
        ),
        # The saved adapter is the checkpoint from this step, not from the step
        # training stopped on. A reader comparing the artifact hashes against
        # the loss curve needs to know which row of the curve they describe.
        "restoredFromStep": held_out_probe.restored_from_step,
        "bestHeldOutStep": held_out_probe.best_step(),
        # Loss says whether the adapter fits the held-out targets; the guards
        # say whether what it generates is usable at all. Recording only the
        # first is how a 56% improvement and a wrong-language answer ended up
        # in the same artifact without contradicting each other.
        "languageMismatchSteps": held_out_probe.language_mismatch_steps(),
        # On the Precision slot this is the guard that does the work: language
        # detection abstains on JSON by design, and degeneracy only fires once a
        # loop is well established. Structural validity caught the same failure
        # twenty steps earlier on the measured curve.
        "jsonInvalidSteps": held_out_probe.json_invalid_steps(),
        # Steps with a loss but no guard result. Nothing here is eligible to be
        # kept, and recording the list makes that auditable rather than implied.
        "stepsWithoutGuardResult": held_out_probe.unevaluated_steps(),
        "bestGuardPassingStep": held_out_probe.best_clean,
        "postRestoreCheck": held_out_probe.post_restore_check,
        "degenerateSteps": [
            entry["step"]
            for entry in held_out_probe.history
            if entry.get("maxDegeneracyScore", 0.0) >= PROBE_DEGENERACY_THRESHOLD
        ],
        "verdict": verdict,
        "baseModelControl": held_out_probe.base_control,
        "history": held_out_probe.history,
        "overheadSeconds": round(held_out_probe.seconds_spent, 1),
    },
    "lora": {
        "r": 8,
        "alpha": 16,
        "dropout": 0.05,
        "targetModules": sorted(lora_config.target_modules),
    },
    "environment": {
        "python": platform.python_version(),
        "torch": torch.__version__,
        "cuda": torch.version.cuda,
        "gpu": torch.cuda.get_device_name(0),
        "computeCapability": list(torch.cuda.get_device_capability(0)),
        "precisionMode": PRECISION_MODE,
        "computeDtype": str(compute_dtype),
        "bf16TensorCores": BF16_TENSOR_CORES,
        "isBf16SupportedReports": torch.cuda.is_bf16_supported(),
        "transformers": transformers.__version__,
        "peft": peft.__version__,
        "trl": trl.__version__,
        "bitsandbytes": bitsandbytes.__version__,
        "datasets": datasets.__version__,
        "accelerate": accelerate.__version__,
        "safetensors": safetensors.__version__,
        "nvidiaSmi": subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"],
            text=True,
        ).strip(),
        "pipFreeze": subprocess.check_output(
            ["python", "-m", "pip", "freeze"], text=True
        ).splitlines(),
    },
    "files": artifact_files,
    "heldOutResults": held_out_results,
}

with open(OUTPUT_DIR / "provenance.json", "w", encoding="utf-8") as handle:
    json.dump(provenance, handle, ensure_ascii=False, indent=2, sort_keys=True)
    handle.write("\n")

# A fixed filename made three successive runs indistinguishable in a downloads
# folder and let a stale copy masquerade as a fresh one. Name the archive after
# what is actually inside it: the slot, the checkpoint step the adapter was
# restored from, and the first eight hex digits of the adapter weights' own
# hash. Two runs can now only collide if they produced identical weights, in
# which case the collision is the correct answer.
adapter_weights_digest = next(
    entry["sha256"]
    for entry in artifact_files
    if entry["path"].endswith("adapter_model.safetensors")
)
archive_step = held_out_probe.restored_from_step
# A file that failed its own checks should say so in its name. Nothing stops a
# zip from being unpacked months later by someone who never reads the
# provenance, and "NOT-PROMOTABLE" in the filename survives that.
archive_name = (
    f"klar-{ADAPTER_SLOT}-adapter"
    f"-s{archive_step if archive_step is not None else 'final'}"
    f"-{adapter_weights_digest[:8]}"
    f"{'' if verdict['promotable'] else '-NOT-PROMOTABLE'}.zip"
)
archive = Path("/kaggle/working") / archive_name
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
    for file_path in sorted(path for path in OUTPUT_DIR.rglob("*") if path.is_file()):
        archive_path = file_path.relative_to(OUTPUT_DIR.parent).as_posix()
        info = zipfile.ZipInfo(archive_path, date_time=(1980, 1, 1, 0, 0, 0))
        info.compress_type = zipfile.ZIP_DEFLATED
        info.external_attr = 0o100644 << 16
        bundle.writestr(info, file_path.read_bytes())
print({
    "archive": str(archive),
    "archive_sha256": file_sha256(archive),
    "adapter_files": artifact_files,
})


## Graduation gate

Download the ZIP and verify its printed SHA-256 before conversion. Convert the
PEFT adapter with the pinned llama.cpp commit documented in LOCAL_LOAD.md, then
load it beside the separately verified Q4_K_M candidate package. Do not promote
either slot on this smoke run alone. Compare base, Precision, Writer, cloud and
deterministic baselines on the frozen multilingual evaluation harness first.
